# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose **Logistic Regression** as my method because it provides a readable, supervised model that answers a simple yes/no question which can be provided by assigning a future decline label of 0 or 1. As such it enables **directional decisions** on which search signals will affect a page's visibility

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a **client-grouped split** (`GroupShuffleSplit` by `client_hash_id`, 70/30). A genuine time-based split isn't buildable yet since I only have one cached month-pair (February → March) — grouping by client is the honest option available now, since multiple pages can belong to the same client and a random split could let the model see a client's other pages during training and then get tested on that same client.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load the same February-March dataset used in Week 4 baseline
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()
print(f"Eligible rows with labels: {len(dataframe)}")

# February features (prior_* columns only — no future information)
X = dataframe[['prior_impressions', 'prior_clicks', 'prior_avg_position',
               'prior_sessions', 'prior_engagement_rate']].copy()
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

model_features = ['prior_impressions', 'prior_clicks', 'prior_ctr',
                   'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
X = X[model_features]
y = dataframe['future_decline_label'].values
groups = dataframe['client_hash_id']

print(f"\nTarget distribution (base rate):")
print(f"  Declined (1): {(y == 1).sum()} rows ({100 * (y == 1).mean():.1f}%)")
print(f"  No decline (0): {(y == 0).sum()} rows ({100 * (y == 0).mean():.1f}%)")

# --- Honest split: grouped by client, so no client's rows appear on both sides ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"\nClient overlap between train/test: {len(overlap)} (should be 0)")
print(f"Train rows: {len(train_idx)} | Test rows: {len(test_idx)}")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
test_dataframe = dataframe.iloc[test_idx].copy()   # test-set rows, for later inspection

# Scale using ONLY training rows — test rows never influence the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression on training rows only
model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
model.fit(X_train_scaled, y_train)
print(f"\nModel trained on {len(X_train)} rows, evaluated on {len(X_test)} held-out rows")

# Score the model ONLY on the held-out test rows
test_dataframe['model_probability'] = model.predict_proba(X_test_scaled)[:, 1]
ranked_by_model = test_dataframe.sort_values('model_probability', ascending=False).reset_index(drop=True)

top_20_model = ranked_by_model.head(20)
precision_at_20_model = top_20_model['future_decline_label'].mean()
base_rate_test = test_dataframe['future_decline_label'].mean()

print(f"\n{'='*60}\nMODEL PERFORMANCE (Logistic Regression, held-out test only)\n{'='*60}")
print(f"Precision@20: {precision_at_20_model:.3f} ({int(top_20_model['future_decline_label'].sum())}/20 actually declined)")
print(f"Base rate (test set): {base_rate_test:.3f}")

# --- Baseline (rule-based, no fitting — but evaluated on the SAME test rows for a fair comparison) ---
test_dataframe['prior_ctr_baseline'] = (
    test_dataframe['prior_clicks'] / test_dataframe['prior_impressions'].replace(0, np.nan)
)
test_dataframe['high_visibility_at_risk'] = (
    (test_dataframe['prior_impressions'] >= 1000) & (test_dataframe['prior_avg_position'] > 10)
).astype(int)
test_dataframe['weak_position_signal'] = (
    test_dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
test_dataframe['low_prior_engagement'] = (
    (test_dataframe['prior_sessions'] > 0) & (test_dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
test_dataframe['low_click_through_rate'] = (
    test_dataframe['prior_ctr_baseline'] < 0.01
).fillna(False).astype(int)
test_dataframe['limited_prior_visibility'] = (
    test_dataframe['prior_impressions'] < 1000
).astype(int)

test_dataframe['baseline_score'] = (
    3 * test_dataframe['limited_prior_visibility']
    + test_dataframe['weak_position_signal']
    + test_dataframe['low_prior_engagement']
    + test_dataframe['low_click_through_rate']
)

ranked_by_baseline = test_dataframe.sort_values(
    ['baseline_score', 'prior_impressions'], ascending=[False, True],
).reset_index(drop=True)
top_20_baseline = ranked_by_baseline.head(20)
precision_at_20_baseline = top_20_baseline['future_decline_label'].mean()

print(f"\n{'='*60}\nBASELINE PERFORMANCE (Rule-based, same test rows)\n{'='*60}")
print(f"Precision@20: {precision_at_20_baseline:.3f} ({int(top_20_baseline['future_decline_label'].sum())}/20 actually declined)")

comparison_table = pd.DataFrame({
    'method': ['Baseline (rule-based)', 'Logistic Regression (model)'],
    'top_20_correct': [
        int(top_20_baseline['future_decline_label'].sum()),
        int(top_20_model['future_decline_label'].sum()),
    ],
    'precision_at_20': [precision_at_20_baseline, precision_at_20_model],
    'base_rate': [base_rate_test, base_rate_test],
    'test_rows': [len(test_dataframe), len(test_dataframe)],
})

print(f"\n{'='*60}\nCOMPARISON TABLE: Baseline vs Model (same held-out test rows)\n{'='*60}")
print(comparison_table.to_string(index=False))

Eligible rows with labels: 80322

Target distribution (base rate):
  Declined (1): 17474 rows (21.8%)
  No decline (0): 62848 rows (78.2%)

Client overlap between train/test: 0 (should be 0)
Train rows: 51418 | Test rows: 28904

Model trained on 51418 rows, evaluated on 28904 held-out rows

MODEL PERFORMANCE (Logistic Regression, held-out test only)
Precision@20: 0.350 (7/20 actually declined)
Base rate (test set): 0.188

BASELINE PERFORMANCE (Rule-based, same test rows)
Precision@20: 0.350 (7/20 actually declined)

COMPARISON TABLE: Baseline vs Model (same held-out test rows)
                     method  top_20_correct  precision_at_20  base_rate  test_rows
      Baseline (rule-based)               7             0.35    0.18769      28904
Logistic Regression (model)               7             0.35    0.18769      28904


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
print("\n" + "="*60)
print("ERROR ANALYSIS: WHERE IS THE MODEL WRONG?")
print("="*60)

# Top 3 features the model relies on (from its learned coefficients)
feature_importance = pd.DataFrame({
    'feature': model_features,
    'coefficient': model.coef_[0],
    'abs_coefficient': np.abs(model.coef_[0]),
})
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print("\n1. TOP 3 FEATURES (model coefficients — what the model relies on):\n")
top_3_features = feature_importance.head(3)
for idx, row in top_3_features.iterrows():
    direction = "increases risk" if row['coefficient'] > 0 else "decreases risk"
    print(f"   {row['feature']}: coefficient = {row['coefficient']:.4f} ({direction})")

print("\nInterpretation check:")
print("  - Do these make sense? (Or are they suspiciously perfect?)")
print("  - prior_impressions: more visibility should... make decline less likely? Check.")
print("  - prior_ctr: higher clicks should make decline less likely? Check.")

# Predictions on the HELD-OUT TEST rows only — never the training rows
test_dataframe['model_prediction'] = model.predict(X_test_scaled)
test_dataframe['model_probability'] = model.predict_proba(X_test_scaled)[:, 1]

test_dataframe['is_error'] = (
    test_dataframe['model_prediction'] != test_dataframe['future_decline_label']
).astype(int)

total_errors = test_dataframe['is_error'].sum()
error_rate = 100 * test_dataframe['is_error'].mean()

print(f"\n2. OVERALL ERROR RATE (on held-out test rows only):")
print(f"   Total errors: {total_errors} / {len(test_dataframe)} ({error_rate:.1f}%)")

print(f"\n3. COMMON PATTERNS IN MODEL ERRORS:")

for label in [0, 1]:
    subset = test_dataframe[test_dataframe['future_decline_label'] == label]
    errors_in_group = subset['is_error'].sum()
    error_pct = 100 * errors_in_group / len(subset) if len(subset) > 0 else 0.0
    label_text = "actually DECLINED" if label == 1 else "did NOT decline"
    print(f"\n   Among rows that {label_text}:")
    print(f"     Model got {errors_in_group}/{len(subset)} wrong ({error_pct:.1f}%)")

    if label == 1 and errors_in_group > 0:
        failing_declines = subset[subset['is_error'] == 1].nlargest(1, 'model_probability')
        if len(failing_declines) > 0:
            row = failing_declines.iloc[0]
            print(f"     Example wrong decline: prior_impressions={row['prior_impressions']:.0f}, "
                  f"prior_ctr={row['prior_ctr_baseline']:.4f}, position={row['prior_avg_position']:.1f}, "
                  f"model_prob={row['model_probability']:.3f} (model said NO decline, but it DID)")

    if label == 0 and errors_in_group > 0:
        failing_non_declines = subset[subset['is_error'] == 1].nsmallest(1, 'model_probability')
        if len(failing_non_declines) > 0:
            row = failing_non_declines.iloc[0]
            print(f"     Example wrong non-decline: prior_impressions={row['prior_impressions']:.0f}, "
                  f"prior_ctr={row['prior_ctr_baseline']:.4f}, position={row['prior_avg_position']:.1f}, "
                  f"model_prob={row['model_probability']:.3f} (model said WILL decline, but it DIDN'T)")

print(f"\n4. CONCRETE WRONG CASES (examples from model errors, test set only):")

errors_df = test_dataframe[test_dataframe['is_error'] == 1].copy()

if len(errors_df) > 0:
    print(f"\n   Case 1: False negative (model missed a decline)")
    false_negatives = test_dataframe[
        (test_dataframe['future_decline_label'] == 1) & (test_dataframe['model_prediction'] == 0)
    ]
    if len(false_negatives) > 0:
        case1 = false_negatives.iloc[0]
        print(f"     - Prior impressions: {case1['prior_impressions']:.0f}")
        print(f"     - Prior CTR: {case1['prior_ctr_baseline']:.4f}")
        print(f"     - Prior position: {case1['prior_avg_position']:.1f}")
        print(f"     - Model probability of decline: {case1['model_probability']:.3f}")
        print(f"     - Reality: page DID decline, but model didn't predict it")

    print(f"\n   Case 2: False positive (model wrongly predicted decline)")
    false_positives = test_dataframe[
        (test_dataframe['future_decline_label'] == 0) & (test_dataframe['model_prediction'] == 1)
    ]
    if len(false_positives) > 0:
        case2 = false_positives.iloc[0]
        print(f"     - Prior impressions: {case2['prior_impressions']:.0f}")
        print(f"     - Prior CTR: {case2['prior_ctr_baseline']:.4f}")
        print(f"     - Prior position: {case2['prior_avg_position']:.1f}")
        print(f"     - Model probability of decline: {case2['model_probability']:.3f}")
        print(f"     - Reality: page did NOT decline, but model predicted it would")

print(f"\n{'='*60}")
print("SUMMARY: What does the model learn and what does it miss?")
print(f"{'='*60}")
print("The model is readable (just 6 feature weights) and learns that prior click")
print(f"metrics and position predict future decline. On held-out clients it misses")
print(f"~{error_rate:.1f}% of cases — a more honest signal than the old in-sample number.")


ERROR ANALYSIS: WHERE IS THE MODEL WRONG?

1. TOP 3 FEATURES (model coefficients — what the model relies on):

   prior_ctr: coefficient = -0.1961 (decreases risk)
   prior_sessions: coefficient = -0.1442 (decreases risk)
   prior_impressions: coefficient = -0.1184 (decreases risk)

Interpretation check:
  - Do these make sense? (Or are they suspiciously perfect?)
  - prior_impressions: more visibility should... make decline less likely? Check.
  - prior_ctr: higher clicks should make decline less likely? Check.

2. OVERALL ERROR RATE (on held-out test rows only):
   Total errors: 5425 / 28904 (18.8%)

3. COMMON PATTERNS IN MODEL ERRORS:

   Among rows that did NOT decline:
     Model got 0/23479 wrong (0.0%)

   Among rows that actually DECLINED:
     Model got 5425/5425 wrong (100.0%)
     Example wrong decline: prior_impressions=113, prior_ctr=0.0000, position=104.7, model_prob=0.429 (model said NO decline, but it DID)

4. CONCRETE WRONG CASES (examples from model errors, test set 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.